## Model Development - Demo Version

In [1]:
# Packages
import pandas as pd
import numpy as np
import os
import tensorflow as tf
import keras
# !pip install torch
# !pip install scikit-learn
# !pip install torch torchvision
# !pip install pandas
# !pip install tensorflow

#### Load data

In [3]:
# Read the data
data_demo = pd.read_excel("../data/data_demo.xlsx")

In [4]:
data_demo.columns

Index(['url', 'filename', 'first_partner_gender', 'first_partner_age',
       'second_partner_gender', 'second_partner_age',
       'first_partner_school_category', 'second_partner_school_category',
       'first_partner_level_id', 'second_partner_level_id',
       'first_partner_field', 'second_partner_field'],
      dtype='object')

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# --- 1. Simulate your demo dataset ---
df = pd.DataFrame({
    'first_partner_gender': ['Female', 'Male', 'Female', 'Female', 'Male'],
    'first_partner_age': ['25-30', '30-35', '25-30', '35-40', '30-35'],
    'first_partner_school_category': ['Ivy', 'Top Public', 'Liberal Arts', 'Intl', 'Other'],
    'second_partner_gender': ['Male', 'Female', 'Male', 'Male', 'Female'],
    'second_partner_age': ['30-35', '25-30', '30-35', '40-45', '25-30'],
    'second_partner_school_category': ['Top Public', 'Ivy', 'Intl', 'Liberal Arts', 'Other']
})

# --- 2. Encode categorical features ---
label_encoders = {}
for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# --- 3. Prepare input and output tensors ---
X = df[['first_partner_gender', 'first_partner_age', 'first_partner_school_category']].values
Y = df[['second_partner_gender', 'second_partner_age', 'second_partner_school_category']].values

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

# --- 4. Define a simple TabTransformer-like model ---
class SimpleTabTransformer(nn.Module):
    def __init__(self, category_sizes, dim, output_sizes):
        super().__init__()
        self.embeds = nn.ModuleList([
            nn.Embedding(size, dim) for size in category_sizes
        ])
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=dim, nhead=2),
            num_layers=2
        )
        self.heads = nn.ModuleList([
            nn.Linear(dim * len(category_sizes), out) for out in output_sizes
        ])

    def forward(self, x):
        emb = [self.embeds[i](x[:, i]) for i in range(x.shape[1])]
        x = torch.stack(emb, dim=1)  # [B, F, D]
        x = self.encoder(x)          # [B, F, D]
        x = x.flatten(start_dim=1)   # [B, F*D]
        return [head(x) for head in self.heads]

# --- 5. Instantiate and test model ---
input_cardinality = [df[col].nunique() for col in ['first_partner_gender', 'first_partner_age', 'first_partner_school_category']]
output_cardinality = [df[col].nunique() for col in ['second_partner_gender', 'second_partner_age', 'second_partner_school_category']]



In [29]:
model = SimpleTabTransformer(input_cardinality, dim=16, output_sizes=output_cardinality)

# --- 6. Forward pass ---
outputs = model(X[:2])
for i, out in enumerate(outputs):
    print(f"Output {i} shape: {out.shape}")  # should be [2, output_classes]


Output 0 shape: torch.Size([2, 2])
Output 1 shape: torch.Size([2, 3])
Output 2 shape: torch.Size([2, 5])


In [ ]:
data_demo = pd.read_excel('/Users/yanmiyu/Desktop/1470_final/DL-Marriage-Prediction/Data/data_demo.xlsx')

In [30]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [31]:
from torch.utils.data import DataLoader, TensorDataset

# Wrap tensors into a dataset and dataloader
dataset = TensorDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Define epochs
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        preds = model(batch_x)
        
        # Each output head → one loss
        losses = [loss_fn(pred, batch_y[:, i]) for i, pred in enumerate(preds)]
        loss = sum(losses)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 11.3204
Epoch 2, Loss: 6.8731
Epoch 3, Loss: 5.4154
Epoch 4, Loss: 4.2870
Epoch 5, Loss: 4.4086
Epoch 6, Loss: 3.2442
Epoch 7, Loss: 2.7964
Epoch 8, Loss: 2.6726
Epoch 9, Loss: 2.4895
Epoch 10, Loss: 2.7951


In [32]:
model.eval()
with torch.no_grad():
    preds = model(X)

# Convert predicted indices to class names
for i, col in enumerate(['second_partner_gender', 'second_partner_age', 'second_partner_school_category']):
    pred_labels = torch.argmax(preds[i], dim=1)
    decoded_preds = label_encoders[col].inverse_transform(pred_labels.numpy())
    true_labels = label_encoders[col].inverse_transform(Y[:, i].numpy())

    print(f"\n=== {col} ===")
    for p, t in zip(decoded_preds, true_labels):
        print(f"Predicted: {p:30}  |  True: {t}")



=== second_partner_gender ===
Predicted: Male                            |  True: Male
Predicted: Female                          |  True: Female
Predicted: Male                            |  True: Male
Predicted: Male                            |  True: Male
Predicted: Female                          |  True: Female

=== second_partner_age ===
Predicted: 30-35                           |  True: 30-35
Predicted: 25-30                           |  True: 25-30
Predicted: 30-35                           |  True: 30-35
Predicted: 40-45                           |  True: 40-45
Predicted: 25-30                           |  True: 25-30

=== second_partner_school_category ===
Predicted: Top Public                      |  True: Top Public
Predicted: Ivy                             |  True: Ivy
Predicted: Intl                            |  True: Intl
Predicted: Liberal Arts                    |  True: Liberal Arts
Predicted: Other                           |  True: Other


### Sample Test - The easiest one to goo...

In [33]:
sample_row = ['Female', '25-30', 'Ivy']

# Step 2: Encode using the same LabelEncoders used in training
sample_encoded = [label_encoders['first_partner_gender'].transform([sample_row[0]])[0],
                  label_encoders['first_partner_age'].transform([sample_row[1]])[0],
                  label_encoders['first_partner_school_category'].transform([sample_row[2]])[0]]

# Convert to tensor: shape [1, 3]
sample_tensor = torch.tensor([sample_encoded], dtype=torch.long)

# Step 3: Pass through model
model.eval()
with torch.no_grad():
    output_preds = model(sample_tensor)

In [34]:
predicted_classes = [torch.argmax(logits, dim=1).item() for logits in output_preds]

decoded_preds = [
    label_encoders['second_partner_gender'].inverse_transform([predicted_classes[0]])[0],
    label_encoders['second_partner_age'].inverse_transform([predicted_classes[1]])[0],
    label_encoders['second_partner_school_category'].inverse_transform([predicted_classes[2]])[0],
]

print("\n=== Predicted Partner Profile ===")
print(f"Gender:         {decoded_preds[0]}")
print(f"Age Group:      {decoded_preds[1]}")
print(f"School Category:{decoded_preds[2]}")



=== Predicted Partner Profile ===
Gender:         Male
Age Group:      30-35
School Category:Top Public


In [ ]:
data_demo = pd.read_excel('/Users/yanmiyu/Desktop/1470_final/DL-Marriage-Prediction/Data/data_demo.xlsx')